# 02 · MIRROR Pipeline Walkthrough

Run the full **Image → Prediction → Evidence → Report** flow on one image and visualise the Grad-CAM overlay inline. Works with ImageNet weights and the offline template report backend — no checkpoint or API key needed.

In [ ]:
import sys, base64, io; sys.path.append('..')
from models.common.config import Config
from models.pipeline import MirrorPipeline

config = Config()  # defaults: densenet121 + gradcam + template report
pipeline = MirrorPipeline(config)

In [ ]:
IMAGE = '../demo/assets/example.png'  # <-- point this at a real radiograph
with open(IMAGE, 'rb') as fh:
    result = pipeline.analyze(fh.read(), modality='chest X-ray')

for f in result.findings[:6]:
    print(f"{'*' if f.present else ' '} {f.label:<20} {f.probability:5.2f}  {f.location}")

## Draft report

In [ ]:
print(result.report)

## Evidence overlay

The Grad-CAM heatmap for the top finding, blended onto the film.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
top = next((f for f in result.findings if f.overlay_png_b64), None)
if top:
    img = Image.open(io.BytesIO(base64.b64decode(top.overlay_png_b64)))
    plt.figure(figsize=(5,5)); plt.imshow(img); plt.axis('off')
    plt.title(f'{top.label} — {top.location}'); plt.show()
else:
    print('No finding above threshold to explain.')